# Yambda 50M likes. NDP


In [ ]:
import gc
import itertools
import json
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG = {
    "data_dir": "../data",
    "interactions_file": "likes.parquet",
    "artist_file": "artist_item_mapping.parquet",
    "album_file": "album_item_mapping.parquet",

    "max_users": None,            
    "min_user_interactions": 3,
    "min_item_interactions": 5,

    "embed_dim": 64,
    "artist_emb_dim": 32,
    "album_emb_dim": 32,
    "tower_hidden": [256, 128, 64],
    "dropout": 0.0,

    "batch_size": 4096,
    "grad_clip": 1.0,
    "lr": 1e-3,
    "weight_decay": 0.0,

    "stage1_epochs": 10,
    "stage2_epochs": 10,

    "resample_head_fraction": 0.20,

    "tune_fast": True,
    "tune_stage1_epochs": 3,
    "tune_stage2_epochs": 3,
    "tune_val_fraction": 1.00,
    "skip_tune_if_cached": True,
    "cache_path": "best_params.json",

    "seeds": [0, 1, 2, 3, 4],

    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 8192,
    "head_fraction": 0.20,

    "seed": 0,
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print(
    f"NDP: stage1={CFG['stage1_epochs']} ep + stage2={CFG['stage2_epochs']} ep, "
    f"{len(CFG['seeds'])} seeds"
)

## 1. Load data

In [ ]:
def find_file(data_dir, name: str) -> Path:
    data_dir = Path(data_dir)
    for p in [data_dir / name, data_dir / f"{name}.parquet"]:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Tried: {[data_dir/name, data_dir/f'{name}.parquet']}")


def normalize_interaction_columns(df: pl.DataFrame) -> pl.DataFrame:
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"uid", "user_id", "userid", "user"}:
            rename[c] = "uid"
        elif lc in {"item_id", "itemid", "track_id", "trackid", "song_id"}:
            rename[c] = "item_id"
        elif lc in {"timestamp", "ts", "time", "event_timestamp", "datetime"}:
            rename[c] = "timestamp"
    return df.rename(rename)


def normalize_item_side_columns(df: pl.DataFrame) -> pl.DataFrame:
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"item_id", "itemid", "track_id", "trackid"}:
            rename[c] = "item_id"
        elif lc in {"artist_id", "artistid"}:
            rename[c] = "artist_id"
        elif lc in {"album_id", "albumid"}:
            rename[c] = "album_id"
    return df.rename(rename)


DATA_DIR = Path(CFG["data_dir"])
INTERACTIONS_PATH = find_file(DATA_DIR, CFG["interactions_file"])
ARTIST_PATH = find_file(DATA_DIR, CFG["artist_file"])
ALBUM_PATH = find_file(DATA_DIR, CFG["album_file"])

print("interactions:", INTERACTIONS_PATH)
print("artists:", ARTIST_PATH)
print("albums:", ALBUM_PATH)

interactions = pl.read_parquet(INTERACTIONS_PATH)
interactions = normalize_interaction_columns(interactions)
print("raw interactions:", interactions.shape)
print("columns:", interactions.columns)

required = {"uid", "item_id"}
missing = required - set(interactions.columns)
assert not missing, f"Missing required columns {missing}. Available: {interactions.columns}"

if "timestamp" not in interactions.columns:
    print("[WARN] timestamp not found: using row index as timestamp")
    interactions = interactions.with_row_index("timestamp")

interactions = interactions.select(["uid", "item_id", "timestamp"])
interactions = (
    interactions
    .sort("timestamp")
    .unique(subset=["uid", "item_id"], keep="first")
)
print("after dedup:", interactions.shape)

## 2. Filtering

In [ ]:
item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
interactions = interactions.join(good_items, on="item_id", how="semi")
print("after item-core:", interactions.shape)

user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
interactions = interactions.join(good_users, on="uid", how="semi")
print("after user-core:", interactions.shape)

if CFG["max_users"] is not None:
    users_sub = (
        interactions.select("uid").unique()
        .sample(n=int(CFG["max_users"]), seed=CFG["seed"])
    )
    interactions = interactions.join(users_sub, on="uid", how="semi")
    print(f"after max_users={CFG['max_users']}:", interactions.shape)

    item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
    good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
    interactions = interactions.join(good_items, on="item_id", how="semi")

    user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
    good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
    interactions = interactions.join(good_users, on="uid", how="semi")

print("final filtered:", interactions.shape)
print("users:", interactions["uid"].n_unique())
print("items:", interactions["item_id"].n_unique())

## 3. ID mapping and leave-one-out split

In [ ]:
user_mapping = interactions.select("uid").unique().sort("uid").with_row_index(name="u_idx")
item_mapping = interactions.select("item_id").unique().sort("item_id").with_row_index(name="i_idx")

inter = (
    interactions
    .join(user_mapping, on="uid", how="inner")
    .join(item_mapping, on="item_id", how="inner")
    .select(["u_idx", "i_idx", "timestamp"])
    .sort(["u_idx", "timestamp"])
)

NUM_USERS = user_mapping.height
NUM_ITEMS = item_mapping.height

inter = inter.with_columns([
    pl.arange(0, pl.len()).over("u_idx").alias("pos"),
    pl.len().over("u_idx").alias("hist_len"),
])

train_df = inter.filter(pl.col("pos") < pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
val_df   = inter.filter(pl.col("pos") == pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
test_df  = inter.filter(pl.col("pos") == pl.col("hist_len") - 1).select(["u_idx", "i_idx"])

train_pairs = train_df.to_numpy().astype(np.int64)
val_np  = val_df.to_numpy().astype(np.int64)
test_np = test_df.to_numpy().astype(np.int64)

val_u,  val_i  = val_np[:, 0],  val_np[:, 1]
test_u, test_i = test_np[:, 0], test_np[:, 1]

print(f"NUM_USERS={NUM_USERS:,}  NUM_ITEMS={NUM_ITEMS:,}")
print(f"train={len(train_pairs):,}  val={len(val_u):,}  test={len(test_u):,}")
print(f"catalog share @50 = {50 / NUM_ITEMS:.6f}")

assert len(train_pairs) > 0
assert len(val_u) == NUM_USERS
assert len(test_u) == NUM_USERS

## 4. Item features: artist_id and album_id

In [ ]:
item_features_df = item_mapping.select(["item_id", "i_idx"])

artists = pl.read_parquet(ARTIST_PATH)
artists = normalize_item_side_columns(artists)
print("artists:", artists.shape, artists.columns)
assert "item_id" in artists.columns and "artist_id" in artists.columns

artists_one = (
    artists.select(["item_id", "artist_id"]).drop_nulls()
    .group_by("item_id").agg(pl.col("artist_id").first())
)
item_features_df = item_features_df.join(artists_one, on="item_id", how="left")

albums = pl.read_parquet(ALBUM_PATH)
albums = normalize_item_side_columns(albums)
print("albums:", albums.shape, albums.columns)
assert "item_id" in albums.columns and "album_id" in albums.columns

albums_one = (
    albums.select(["item_id", "album_id"]).drop_nulls()
    .group_by("item_id").agg(pl.col("album_id").first())
)
item_features_df = item_features_df.join(albums_one, on="item_id", how="left")

item_features_df = item_features_df.sort("i_idx")

unique_artists = (
    item_features_df.select("artist_id").drop_nulls().unique()
    .sort("artist_id").with_row_index(name="artist_idx", offset=1)
)
item_features_df = (
    item_features_df.join(unique_artists, on="artist_id", how="left")
    .with_columns(pl.col("artist_idx").fill_null(0).cast(pl.Int64))
)

unique_albums = (
    item_features_df.select("album_id").drop_nulls().unique()
    .sort("album_id").with_row_index(name="album_idx", offset=1)
)
item_features_df = (
    item_features_df.join(unique_albums, on="album_id", how="left")
    .with_columns(pl.col("album_idx").fill_null(0).cast(pl.Int64))
)

ITEM_ARTIST = item_features_df["artist_idx"].to_numpy().astype(np.int64)
ITEM_ALBUM  = item_features_df["album_idx"].to_numpy().astype(np.int64)
NUM_ARTISTS = int(ITEM_ARTIST.max()) + 1
NUM_ALBUMS  = int(ITEM_ALBUM.max()) + 1

print(f"NUM_ARTISTS={NUM_ARTISTS}  NUM_ALBUMS={NUM_ALBUMS}")
print(f"artist unknown share: {float((ITEM_ARTIST == 0).mean()):.4f}")
print(f"album  unknown share: {float((ITEM_ALBUM  == 0).mean()):.4f}")

item_artist_t = torch.from_numpy(ITEM_ARTIST).long().to(device)
item_album_t  = torch.from_numpy(ITEM_ALBUM).long().to(device)

## 5. Known items and head/tail split

In [ ]:
train_user_items = [set() for _ in range(NUM_USERS)]
for u, i in train_pairs:
    train_user_items[int(u)].add(int(i))

known_val  = [set(s) for s in train_user_items]
known_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)
order     = np.argsort(-item_freq)

n_head    = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,}  tail={(~head_mask).sum():,}")
print(f"nonzero train items: {np.sum(item_freq > 0):,}")
print(f"zero train items:    {np.sum(item_freq == 0):,}")
print(f"val  true head share: {head_mask[val_i].mean():.4f}")
print(f"test true head share: {head_mask[test_i].mean():.4f}")

## 6. Dataset and model

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list, dropout: float = 0.0):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x):
        return self.net(x)


class TwoTowerNDP(nn.Module):

    def __init__(
        self,
        num_users: int,
        num_items: int,
        num_artists: int,
        num_albums: int,
        embed_dim: int = 64,
        artist_emb_dim: int = 32,
        album_emb_dim: int = 32,
        hidden: list = [256, 128, 64],
        dropout: float = 0.0,
    ):
        super().__init__()
        self.user_emb   = nn.Embedding(num_users, embed_dim)
        self.item_emb   = nn.Embedding(num_items, embed_dim)
        self.artist_emb = nn.Embedding(num_artists, artist_emb_dim)
        self.album_emb  = nn.Embedding(num_albums, album_emb_dim)

        self.user_mlp = MLP(embed_dim, hidden, dropout)
        self.item_mlp = MLP(embed_dim + artist_emb_dim + album_emb_dim, hidden, dropout)
        self.user_ln  = nn.LayerNorm(self.user_mlp.out_dim)
        self.item_ln  = nn.LayerNorm(self.item_mlp.out_dim)

        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_emb, self.item_emb, self.artist_emb, self.album_emb]:
            nn.init.normal_(emb.weight, std=0.01)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_vec(self, u: torch.Tensor) -> torch.Tensor:
        return self.user_ln(self.user_mlp(self.user_emb(u)))

    def item_vec(self, i: torch.Tensor) -> torch.Tensor:
        x = torch.cat([
            self.item_emb(i),
            self.artist_emb(item_artist_t[i]),
            self.album_emb(item_album_t[i]),
        ], dim=-1)
        return self.item_ln(self.item_mlp(x))


def inbatch_softmax_loss(user_vecs: torch.Tensor, item_vecs: torch.Tensor) -> torch.Tensor:
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)
    logits = u @ v.T
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


def make_train_loader(pairs: np.ndarray, batch_size: int, shuffle: bool = True) -> DataLoader:
    return DataLoader(
        PairDataset(pairs),
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_train_loader(train_pairs, CFG["batch_size"])
print("train loader ready,", len(train_loader), "batches/epoch")

## 7. Re-balanced dataset for NDP Stage 2

In [ ]:
def build_rebalanced_pairs(
    train_pairs: np.ndarray,
    item_freq: np.ndarray,
    head_fraction: float,
    rng_seed: int = 0,
) -> np.ndarray:

    n_items = len(item_freq)
    order   = np.argsort(-item_freq)
    n_head  = max(1, int(n_items * head_fraction))

    local_head_mask = np.zeros(n_items, dtype=bool)
    local_head_mask[order[:n_head]] = True

    tail_freq = item_freq[~local_head_mask]
    target_freq = int(tail_freq.max()) if tail_freq.max() > 0 else 1

    rng = np.random.default_rng(rng_seed)

    kept = []
    item_to_rows = {}
    for row_idx, (u, i) in enumerate(train_pairs):
        item_to_rows.setdefault(int(i), []).append(row_idx)

    for item_idx, row_indices in item_to_rows.items():
        if local_head_mask[item_idx]:
            n_keep = min(target_freq, len(row_indices))
            chosen = rng.choice(row_indices, size=n_keep, replace=False)
            kept.extend(chosen.tolist())
        else:
            kept.extend(row_indices)

    rebalanced = train_pairs[sorted(kept)]
    return rebalanced


rebalanced_pairs = build_rebalanced_pairs(
    train_pairs=train_pairs,
    item_freq=item_freq,
    head_fraction=CFG["resample_head_fraction"],
    rng_seed=CFG["seed"],
)

rebalanced_loader = make_train_loader(rebalanced_pairs, CFG["batch_size"])

rb_freq = np.bincount(rebalanced_pairs[:, 1], minlength=NUM_ITEMS)
print(f"Original  train pairs: {len(train_pairs):,}")
print(f"Rebalanced train pairs: {len(rebalanced_pairs):,}")
print(f"Original  head/tail freq ratio: {item_freq[head_mask].mean():.1f} / {item_freq[~head_mask].mean():.1f}")
print(f"Rebalanced head/tail freq ratio: {rb_freq[head_mask].mean():.1f} / {rb_freq[~head_mask].mean():.1f}")

orig_if = item_freq.max() / item_freq[item_freq > 0].min()
rb_nonzero = rb_freq[rb_freq > 0]
rb_if = rb_freq.max() / rb_nonzero.min() if len(rb_nonzero) else float("nan")
print(f"Imbalance factor original:   {orig_if:.1f}")
print(f"Imbalance factor rebalanced: {rb_if:.1f}")

## 8. Evaluation

In [ ]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 8192,
) -> Dict:
    model.eval()

    ranks_all, ranks_head, ranks_tail = [], [], []
    recommended_by_k = {k: [] for k in ks}
    max_k = max(ks)

    item_vectors = []
    for s in tqdm(range(0, NUM_ITEMS, item_batch_size), desc="item vecs", leave=False):
        e   = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)
        v   = F.normalize(model.item_vec(idx), dim=-1, eps=1e-6)
        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors [{s}:{e}]")
        item_vectors.append(v.cpu())
    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    for start in tqdm(range(0, len(users), user_batch_size), desc="eval users", leave=False):
        end = min(start + user_batch_size, len(users))
        bu  = users[start:end]
        bi  = true_items[start:end]

        ut    = torch.tensor(bu, device=device, dtype=torch.long)
        uvec  = F.normalize(model.user_vec(ut), dim=-1, eps=1e-6)
        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors [{start}:{end}]")

        scores = (uvec @ item_vectors.T).cpu().numpy()
        if not np.isfinite(scores).all():
            raise RuntimeError(f"Non-finite scores [{start}:{end}]")

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u, true_i = int(u), int(true_i)
            srow = scores[row].copy()

            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9

            true_score = srow[true_i]
            eps = 1e-12
            num_greater = int((srow > true_score + eps).sum())
            num_tied    = int((np.abs(srow - true_score) <= eps).sum())
            rank = num_greater + num_tied - 1

            ranks_all.append(rank)
            (ranks_head if head_mask[true_i] else ranks_tail).append(rank)

            if max_k < len(srow):
                top_un  = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_un[np.argsort(-srow[top_un])]
            else:
                top_idx = np.argsort(-srow)
            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    def agg_accuracy(ranks: list) -> dict:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}
        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k
                out[f"HR@{k}"]   = 100.0 * hits.mean()
                out[f"NDCG@{k}"] = 100.0 * np.where(hits, 1.0 / np.log2(a + 2), 0.0).mean()
        return out

    tail_mask = ~head_mask
    num_tail_items  = int(tail_mask.sum())
    popularity_log  = np.log1p(item_freq.astype(np.float64))
    long_tail_metrics = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])
        unique_recs = np.unique(recs)

        catalog_coverage = len(unique_recs) / NUM_ITEMS
        tail_coverage    = (np.sum(tail_mask[unique_recs]) / num_tail_items
                            if num_tail_items > 0 else np.nan)
        avg_popularity   = popularity_log[recs].mean()
        tail_ratio       = tail_mask[recs].mean()

        n_eval = recs.shape[0]
        if n_eval <= 1:
            personalization = np.nan
        else:
            exposure = np.bincount(recs.reshape(-1), minlength=NUM_ITEMS)
            pairwise = np.sum(exposure * (exposure - 1) / 2)
            num_pairs = n_eval * (n_eval - 1) / 2
            personalization = 1.0 - (pairwise / num_pairs) / k

        long_tail_metrics[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail_metrics[f"TailCoverage@{k}"]    = 100.0 * tail_coverage
        long_tail_metrics[f"AvgPopularity@{k}"]   = float(avg_popularity)
        long_tail_metrics[f"TailRatio@{k}"]        = 100.0 * tail_ratio
        long_tail_metrics[f"Personalization@{k}"]  = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head":    agg_accuracy(ranks_head),
        "tail":    agg_accuracy(ranks_tail),
        "long_tail": long_tail_metrics,
        "counts": {
            "overall": len(ranks_all),
            "head":    len(ranks_head),
            "tail":    len(ranks_tail),
        },
    }


def print_metrics(metrics: dict):
    print("counts:", metrics.get("counts", {}))
    for split in ["overall", "head", "tail"]:
        print(f"[{split}]", metrics[split])
    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])


def make_head_mask(item_freq: np.ndarray, head_fraction: float) -> np.ndarray:
    n_head = max(1, int(len(item_freq) * head_fraction))
    order  = np.argsort(-item_freq)
    mask   = np.zeros(len(item_freq), dtype=bool)
    mask[order[:n_head]] = True
    return mask


@torch.no_grad()
def evaluate_head_tail_sweep(
    model: nn.Module,
    method_name: str,
    seed: int,
    head_fractions: List[float],
    test_u: np.ndarray,
    test_i: np.ndarray,
    known_test: List[set],
    item_freq: np.ndarray,
    ks: List[int],
) -> List[dict]:
    rows = []
    model.eval()
    for hf in head_fractions:
        print(f"\n{'='*80}")
        print(f"{method_name} | seed={seed} | head_fraction={hf} ({100*hf:.3f}%)")
        print(f"{'='*80}")

        cur_mask   = make_head_mask(item_freq, hf)
        n_head     = int(cur_mask.sum())
        n_tail     = int((~cur_mask).sum())
        head_share = float(cur_mask[test_i].mean())
        tail_share = float((~cur_mask[test_i]).mean())

        print(f"head items={n_head:,}  tail items={n_tail:,}")
        print(f"test head share={head_share:.4f}  tail share={tail_share:.4f}")

        metrics = evaluate_full_corpus(
            model=model,
            users=test_u,
            true_items=test_i,
            known_user_items=known_test,
            head_mask=cur_mask,
            ks=ks,
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )
        print_metrics(metrics)

        row = {
            "method": method_name,
            "seed": seed,
            "head_fraction": hf,
            "head_percent": 100.0 * hf,
            "num_head_items": n_head,
            "num_tail_items": n_tail,
            "test_head_share": head_share,
            "test_tail_share": tail_share,
        }
        for split in ("overall", "head", "tail"):
            for key, val in metrics[split].items():
                row[f"{split}_{key}"] = val
        if "long_tail" in metrics:
            for key, val in metrics["long_tail"].items():
                row[key] = val
        if "counts" in metrics:
            for key, val in metrics["counts"].items():
                row[f"count_{key}"] = val
        rows.append(row)
    return rows

## 9. NDP training routine

In [ ]:
def run_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loader: DataLoader,
    epoch: int,
    total_epochs: int,
    desc_prefix: str = "",
) -> float:
    model.train()
    total_loss, total_n = 0.0, 0

    pbar = tqdm(loader, desc=f"{desc_prefix}ep{epoch}/{total_epochs}", leave=False)
    for users_batch, items_batch in pbar:
        users_batch = users_batch.to(device, non_blocking=True)
        items_batch = items_batch.to(device, non_blocking=True)

        user_vecs = model.user_vec(users_batch)
        item_vecs = model.item_vec(items_batch)

        loss = inbatch_softmax_loss(user_vecs, item_vecs)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss.item()}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()

        bs = users_batch.size(0)
        total_loss += loss.item() * bs
        total_n    += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(total_n, 1)


def train_ndp(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    stage1_epochs: int,
    stage2_epochs: int,
    val_u: np.ndarray,
    val_i: np.ndarray,
    known_val: List[set],
    head_mask: np.ndarray,
    item_freq: np.ndarray,
    seed_tag: str = "",
    track_val: bool = True,
) -> tuple:

    best_val_hr50 = -1.0
    best_state    = None
    history       = []

    k_track = CFG["eval_k"][-1]

    print(f"\n[NDP Stage 1] {stage1_epochs} epochs on original distribution")
    for ep in range(1, stage1_epochs + 1):
        avg_loss = run_one_epoch(
            model, optimizer, train_loader, ep, stage1_epochs,
            desc_prefix=f"[S1 {seed_tag}] ",
        )
        log = {"stage": 1, "epoch": ep, "train_loss": avg_loss}

        if track_val:
            vm = evaluate_full_corpus(
                model=model,
                users=val_u,
                true_items=val_i,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )
            hr = vm["overall"].get(f"HR@{k_track}", -1.0)
            log["val_HR@50"] = hr
            if hr > best_val_hr50:
                best_val_hr50 = hr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  [S1] ep{ep} loss={avg_loss:.4f}  val HR@{k_track}={hr:.4f}")
        else:
            print(f"  [S1] ep{ep} loss={avg_loss:.4f}")

        history.append(log)

    print(f"\n[NDP Stage 2] {stage2_epochs} epochs on rebalanced distribution")
    for ep in range(1, stage2_epochs + 1):
        avg_loss = run_one_epoch(
            model, optimizer, rebalanced_loader, ep, stage2_epochs,
            desc_prefix=f"[S2 {seed_tag}] ",
        )
        log = {"stage": 2, "epoch": ep, "train_loss": avg_loss}

        if track_val:
            vm = evaluate_full_corpus(
                model=model,
                users=val_u,
                true_items=val_i,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )
            hr = vm["overall"].get(f"HR@{k_track}", -1.0)
            log["val_HR@50"] = hr
            if hr > best_val_hr50:
                best_val_hr50 = hr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  [S2] ep{ep} loss={avg_loss:.4f}  val HR@{k_track}={hr:.4f}")
        else:
            print(f"  [S2] loss={avg_loss:.4f}")

        history.append(log)

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_hr50, history

## 10. Grid search

In [ ]:
LR_GRID = [0.001]
DROPOUT_GRID = [0.1]
WD_GRID = [0.0]

combos  = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID))
k_main  = CFG["eval_k"][-1]
cache_path = Path(CFG["cache_path"])
lb_csv = cache_path.with_suffix(".leaderboard.csv")

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")
    leaderboard_df = pd.read_csv(lb_csv) if lb_csv.exists() else None
else:
    frac = float(CFG.get("tune_val_fraction", 1.0))
    if frac < 1.0 and CFG["tune_fast"]:
        n_tune = max(1, int(len(val_u) * frac))
        _idx   = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[_idx], val_i[_idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    s1_ep = CFG["tune_stage1_epochs"]
    s2_ep = CFG["tune_stage2_epochs"]

    print(
        f"Tuning {len(combos)} trials × (S1={s1_ep} + S2={s2_ep}) ep  "
        f"val subset: {len(val_u_t):,}/{len(val_u):,}"
    )

    best_hr, best_hp = -1.0, None
    leaderboard = []

    for ti, (lr, dr, wd) in enumerate(combos, 1):
        set_seed(CFG["seed"])

        m = TwoTowerNDP(
            NUM_USERS, NUM_ITEMS, NUM_ARTISTS, NUM_ALBUMS,
            embed_dim=CFG["embed_dim"],
            artist_emb_dim=CFG["artist_emb_dim"],
            album_emb_dim=CFG["album_emb_dim"],
            hidden=CFG["tower_hidden"],
            dropout=dr,
        ).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)

        status, error_msg = "ok", ""
        hr = -1.0
        met = None

        try:
            best_state_t, best_hr_t, hist_t = train_ndp(
                model=m, optimizer=opt,
                stage1_epochs=s1_ep, stage2_epochs=s2_ep,
                val_u=val_u_t, val_i=val_i_t,
                known_val=known_val,
                head_mask=head_mask, item_freq=item_freq,
                seed_tag=f"t{ti}",
                track_val=True,
            )
            hr = best_hr_t
        except RuntimeError as e:
            status, error_msg = "failed", str(e)
            print(f"  trial {ti} FAILED: {e}")

        row = {
            "trial": ti, "status": status, "error": error_msg,
            "lr": lr, "dropout": dr, "weight_decay": wd,
            "tune_s1_epochs": s1_ep, "tune_s2_epochs": s2_ep,
            "val_subset_size": len(val_u_t), "val_full_size": len(val_u),
            f"val_HR@{k_main}": hr,
        }
        leaderboard.append(row)
        print(f"  trial {ti:3d}/{len(combos)}  lr={lr:.0e} dr={dr} wd={wd:.0e} -> val HR@{k_main}={hr:.2f}%")

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {"lr": lr, "dropout": dr, "weight_decay": wd, f"val_HR@{k_main}": hr}

        del m, opt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    leaderboard_df = pd.DataFrame(leaderboard).sort_values(f"val_HR@{k_main}", ascending=False, na_position="last")
    leaderboard_df.to_csv(lb_csv, index=False)

    if best_hp is None:
        raise RuntimeError("No successful trial. Check leaderboard.")

    cache_path.write_text(json.dumps(best_hp, indent=2))
    print(f"\nBest val HR@{k_main}={best_hr:.2f}%  ->  {best_hp}")
    print(f"Saved: {cache_path}  |  leaderboard: {lb_csv}")

leaderboard_df.head(10) if leaderboard_df is not None else None

## 11. Final training

In [ ]:
HEAD_FRACTIONS = [0.001, 0.005, 0.01, 0.05, 0.20]

all_rows       = []
all_test       = []
all_sweep_rows = []

print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print(f"\n{'='*80}")
    print(f"NDP  seed={seed}")
    print(f"{'='*80}")

    set_seed(seed)

    model = TwoTowerNDP(
        NUM_USERS, NUM_ITEMS, NUM_ARTISTS, NUM_ALBUMS,
        embed_dim=CFG["embed_dim"],
        artist_emb_dim=CFG["artist_emb_dim"],
        album_emb_dim=CFG["album_emb_dim"],
        hidden=CFG["tower_hidden"],
        dropout=best_hp["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_hp["lr"],
        weight_decay=best_hp["weight_decay"],
    )

    best_state, best_val_hr50, history = train_ndp(
        model=model, optimizer=optimizer,
        stage1_epochs=CFG["stage1_epochs"],
        stage2_epochs=CFG["stage2_epochs"],
        val_u=val_u, val_i=val_i,
        known_val=known_val,
        head_mask=head_mask, item_freq=item_freq,
        seed_tag=f"seed{seed}",
        track_val=True,
    )

    print(f"\nseed {seed} best val HR@50: {best_val_hr50:.4f}")

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u, true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask, ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )
    print("TEST METRICS")
    print_metrics(test_metrics)
    all_test.append(test_metrics)

    row = {
        "method": "NDP",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "stage1_epochs": CFG["stage1_epochs"],
        "stage2_epochs": CFG["stage2_epochs"],
        "best_val_HR@50": best_val_hr50,
    }
    for split in ("overall", "head", "tail"):
        for key, val in test_metrics[split].items():
            row[f"test_{split}_{key}"] = val
    if "long_tail" in test_metrics:
        for key, val in test_metrics["long_tail"].items():
            row[f"test_{key}"] = val
    if "counts" in test_metrics:
        for key, val in test_metrics["counts"].items():
            row[f"test_count_{key}"] = val
    all_rows.append(row)

    sweep_rows = evaluate_head_tail_sweep(
        model=model,
        method_name="NDP",
        seed=seed,
        head_fractions=HEAD_FRACTIONS,
        test_u=test_u, test_i=test_i,
        known_test=known_test,
        item_freq=item_freq,
        ks=CFG["eval_k"],
    )
    all_sweep_rows.extend(sweep_rows)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


metrics_df = pd.DataFrame(all_rows)
sweep_df = pd.DataFrame(all_sweep_rows)

metrics_df

## 12. Summary table

In [ ]:
def make_final_summary_table(
    metrics_df: pd.DataFrame,
    sweep_df: pd.DataFrame,
    method_name: str = "NDP",
    save_path: str | None = None,
) -> pd.DataFrame:

    if len(sweep_df) == 0:
        raise ValueError("sweep_df is empty")

    selected_metrics = [
        "overall_HR@50",
        "overall_NDCG@50",
        "head_HR@50",
        "head_NDCG@50",
        "tail_HR@50",
        "tail_NDCG@50",
        "CatalogCoverage@50",
        "TailCoverage@50",
        "AvgPopularity@50",
        "TailRatio@50",
        "Personalization@50",
    ]

    rows = []
    for head_fraction, group in sweep_df.groupby("head_fraction"):
        group = group.copy()
        row = {
            "method": method_name,
            "head_share": f"{100 * float(head_fraction):.3f}%",
            "head_fraction": float(head_fraction),
            "num_seeds": group["seed"].nunique() if "seed" in group.columns else len(group),
            "num_head_items": int(group["num_head_items"].iloc[0]) if "num_head_items" in group.columns else np.nan,
            "num_tail_items": int(group["num_tail_items"].iloc[0]) if "num_tail_items" in group.columns else np.nan,
        }
        if "test_head_share" in group.columns:
            row["test_head_share"] = f"{100 * float(group['test_head_share'].mean()):.2f}%"
        if "test_tail_share" in group.columns:
            row["test_tail_share"] = f"{100 * float(group['test_tail_share'].mean()):.2f}%"

        if metrics_df is not None and len(metrics_df) > 0:
            for hp_col in ["lr", "dropout", "weight_decay", "stage1_epochs", "stage2_epochs"]:
                if hp_col in metrics_df.columns:
                    vals = metrics_df[hp_col].dropna().unique()
                    row[hp_col] = vals[0] if len(vals) == 1 else ", ".join(map(str, vals))
            if "best_val_HR@50" in metrics_df.columns:
                vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
                if len(vals):
                    m = np.mean(vals)
                    s = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
                    row["best_val_HR@50"] = f"{m:.2f} ± {s:.2f}"

        for metric in selected_metrics:
            if metric not in group.columns:
                continue
            vals = group[metric].dropna().to_numpy(dtype=float)
            if not len(vals):
                row[metric] = "nan"
                continue
            m = np.mean(vals)
            s = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
            fmt = f"{m:.4f} ± {s:.4f}" if "AvgPopularity" in metric else f"{m:.2f} ± {s:.2f}"
            row[metric] = fmt

        rows.append(row)

    first_cols = [
        "method", "head_share", "head_fraction", "num_seeds",
        "num_head_items", "num_tail_items", "test_head_share", "test_tail_share",
        "lr", "dropout", "weight_decay", "stage1_epochs", "stage2_epochs", "best_val_HR@50",
    ]
    metric_cols = [
        "overall_HR@50", "overall_NDCG@50",
        "head_HR@50", "head_NDCG@50",
        "tail_HR@50", "tail_NDCG@50",
        "CatalogCoverage@50", "TailCoverage@50",
        "AvgPopularity@50", "TailRatio@50", "Personalization@50",
    ]
    ordered = [c for c in first_cols + metric_cols if c in pd.DataFrame(rows).columns]
    summary_df = pd.DataFrame(rows).sort_values("head_fraction").reset_index(drop=True)
    other = [c for c in summary_df.columns if c not in ordered]
    summary_df = summary_df[ordered + other]

    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")

    return summary_df

In [ ]:
summary_df = make_final_summary_table(
    metrics_df, sweep_df,
    method_name="NDP",
    save_path="summary.csv",
)
summary_df

## 13. Aggregate test results

In [ ]:
print("\n" + "=" * 70)
print("NDP — TEST  mean ± stderr (default head_fraction=20%)")
print("=" * 70)
for split in ("overall", "head", "tail"):
    for mk in [f"HR@{k}" for k in CFG["eval_k"]] + [f"NDCG@{k}" for k in CFG["eval_k"]]:
        vals = [r[split][mk] for r in all_test if np.isfinite(r[split][mk])]
        if not vals:
            print(f"  [{split}] {mk}: nan")
            continue
        mean = np.mean(vals)
        se   = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
        print(f"  [{split}] {mk}: {mean:.2f} ± {se:.2f}")